# Binary Classification with Neural Networks on the Census Income Dataset

**Workshop implementation using PyTorch**

### Objective
Build a binary classification neural network that predicts whether an individual earns **more than $50,000 annually**.

- Dataset: 30,000 records
- Training set: 25,000 records
- Testing set: 5,000 records
- Framework: PyTorch
- Hidden layer: 50 neurons
- Dropout: 0.4
- Loss: CrossEntropyLoss
- Optimizer: Adam (learning rate = 0.001)
- Epochs: 300


## 1. Import Libraries

We use pandas for data handling, NumPy for arrays, PyTorch for the neural network, and scikit-learn for preprocessing and evaluation.

In [ ]:
import os
os.environ["OPENBLAS_NUM_THREADS"] = "1"

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report


## 2. Load the Dataset

In [ ]:
df = pd.read_csv("../data/income.csv")

print("Dataset shape:", df.shape)
display(df.head())


## 3. Separate Categorical, Continuous, and Label Columns

Categorical features are converted into integer category codes so they can be passed to embedding layers.

Continuous features are:
- `age`
- `education-num`
- `hours-per-week`

Categorical features are:
- `sex`
- `education`
- `marital-status`
- `workclass`
- `occupation`

The `label` column is the binary target. The text column `income` is not used as an input because it directly represents the target.

In [ ]:
cat_cols = ["sex", "education", "marital-status", "workclass", "occupation"]
cont_cols = ["age", "education-num", "hours-per-week"]
label_col = "label"

print("Categorical columns:", cat_cols)
print("Continuous columns:", cont_cols)
print("Label column:", label_col)


In [ ]:
# Convert categorical columns to integer codes
cat_arrays = []
cat_sizes = []
category_maps = {}

for col in cat_cols:
    values, uniques = pd.factorize(df[col].fillna("Unknown").astype(str), sort=True)
    cat_arrays.append(values.astype(np.int64))
    cat_sizes.append(len(uniques))
    category_maps[col] = {str(v): int(i) for i, v in enumerate(uniques)}

# Continuous values and labels
cont_array = df[cont_cols].fillna(df[cont_cols].median()).values.astype(np.float32)
labels = df[label_col].values.astype(np.int64)

print("Categorical array shape:", np.stack(cat_arrays, axis=1).shape)
print("Continuous array shape:", cont_array.shape)
print("Label array shape:", labels.shape)
print("Embedding category sizes:", cat_sizes)


## 4. Create a Reproducible 25,000 / 5,000 Train-Test Split

In [ ]:
SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# Shuffle indices reproducibly, then take exactly 25,000 for training and 5,000 for testing
rng = np.random.default_rng(SEED)
indices = rng.permutation(len(df))

train_idx = indices[:25000]
test_idx = indices[25000:]

cat_data = np.stack(cat_arrays, axis=1)
X_cat_train = cat_data[train_idx]
X_cat_test = cat_data[test_idx]

X_cont_train = cont_array[train_idx]
X_cont_test = cont_array[test_idx]

y_train = labels[train_idx]
y_test = labels[test_idx]

print("Training samples:", len(train_idx))
print("Testing samples:", len(test_idx))


## 5. Scale Continuous Features

The continuous variables are standardized using statistics from the training set only. This avoids using test-set information during preprocessing.

In [ ]:
scaler = StandardScaler()

X_cont_train = scaler.fit_transform(X_cont_train).astype(np.float32)
X_cont_test = scaler.transform(X_cont_test).astype(np.float32)

# Convert arrays to PyTorch tensors
X_cat_train = torch.tensor(X_cat_train, dtype=torch.long)
X_cat_test = torch.tensor(X_cat_test, dtype=torch.long)

X_cont_train = torch.tensor(X_cont_train, dtype=torch.float32)
X_cont_test = torch.tensor(X_cont_test, dtype=torch.float32)

y_train = torch.tensor(y_train, dtype=torch.long)
y_test = torch.tensor(y_test, dtype=torch.long)

print("Categorical tensor:", X_cat_train.shape)
print("Continuous tensor:", X_cont_train.shape)
print("Label tensor:", y_train.shape)


## 6. Define the Tabular Neural Network

The model:
1. Creates an embedding for every categorical feature.
2. Batch-normalizes continuous features.
3. Concatenates all representations.
4. Uses **one hidden layer with 50 neurons**.
5. Uses **dropout p = 0.4**.
6. Produces two output values for the two classes: `<=50K` and `>50K`.

In [ ]:
class TabularModel(nn.Module):
    def __init__(self, emb_sizes, n_cont, hidden=50, dropout=0.4):
        super().__init__()

        self.embeddings = nn.ModuleList([
            nn.Embedding(size, min(50, (size + 1) // 2))
            for size in emb_sizes
        ])

        emb_total = sum(emb.embedding_dim for emb in self.embeddings)

        self.batch_norm = nn.BatchNorm1d(n_cont)

        self.network = nn.Sequential(
            nn.Linear(emb_total + n_cont, hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, 2)
        )

    def forward(self, x_cat, x_cont):
        embedded = [
            emb(x_cat[:, i])
            for i, emb in enumerate(self.embeddings)
        ]

        x = torch.cat(embedded + [self.batch_norm(x_cont)], dim=1)
        return self.network(x)


In [ ]:
model = TabularModel(
    emb_sizes=cat_sizes,
    n_cont=len(cont_cols),
    hidden=50,
    dropout=0.4
)

print(model)


## 7. Set Loss Function and Adam Optimizer

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

print("Loss function:", criterion)
print("Optimizer:", optimizer.__class__.__name__)
print("Learning rate: 0.001")


## 8. Train for 300 Epochs

In [ ]:
epochs = 300
train_losses = []

for epoch in range(epochs):
    model.train()

    optimizer.zero_grad()
    output = model(X_cat_train, X_cont_train)
    loss = criterion(output, y_train)

    loss.backward()
    optimizer.step()

    train_losses.append(loss.item())

    if (epoch + 1) % 50 == 0:
        print(f"Epoch {epoch + 1:3d}/{epochs} - Loss: {loss.item():.4f}")


Epoch  50/300 - Loss: 0.2926
Epoch 100/300 - Loss: 0.2782
Epoch 150/300 - Loss: 0.2699
Epoch 200/300 - Loss: 0.2661
Epoch 250/300 - Loss: 0.2636
Epoch 300/300 - Loss: 0.2631


## 9. Plot Training Loss

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(train_losses)
plt.xlabel("Epoch")
plt.ylabel("Training Loss")
plt.title("Training Loss over 300 Epochs")
plt.show()


## 10. Evaluate on the Test Set

In [ ]:
model.eval()

with torch.no_grad():
    test_output = model(X_cat_test, X_cont_test)
    test_loss = criterion(test_output, y_test).item()
    predictions = test_output.argmax(dim=1)

accuracy = accuracy_score(y_test.numpy(), predictions.numpy())

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {accuracy * 100:.2f}%")


Test Loss: 0.2670
Test Accuracy: 88.02%


## 11. Confusion Matrix and Classification Report

In [ ]:
cm = confusion_matrix(y_test.numpy(), predictions.numpy())

print("Confusion Matrix:")
print(cm)

print("\nClassification Report:")
print(classification_report(
    y_test.numpy(),
    predictions.numpy(),
    target_names=["<=50K", ">50K"]
))


Confusion Matrix:
[[3160  414]
 [ 185 1241]]

Classification Report:
              precision    recall  f1-score   support

       <=50K       0.94      0.88      0.91      3574
        >50K       0.75      0.87      0.81      1426

    accuracy                           0.88      5000
   macro avg       0.85      0.88      0.86      5000
weighted avg       0.89      0.88      0.88      5000


In [ ]:
plt.figure(figsize=(5, 4))
plt.imshow(cm)
plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.xticks([0, 1], ["<=50K", ">50K"])
plt.yticks([0, 1], ["<=50K", ">50K"])

for i in range(2):
    for j in range(2):
        plt.text(j, i, cm[i, j], ha="center", va="center")

plt.show()


## 12. Bonus: Predict for a New Individual

The function below accepts the same categorical and continuous features used by the model and returns the predicted income class.

In [ ]:
def predict_income(
    age, sex, education, education_num,
    marital_status, workclass, occupation, hours_per_week
):
    model.eval()

    row = {
        "age": age,
        "sex": sex,
        "education": education,
        "education-num": education_num,
        "marital-status": marital_status,
        "workclass": workclass,
        "occupation": occupation,
        "hours-per-week": hours_per_week
    }

    cat_values = []
    for col in cat_cols:
        value = str(row[col])
        if value not in category_maps[col]:
            raise ValueError(f"Unknown value for {col}: {value}")
        cat_values.append(category_maps[col][value])

    cont_values = np.array([[row[c] for c in cont_cols]], dtype=np.float32)
    cont_values = scaler.transform(cont_values).astype(np.float32)

    cat_tensor = torch.tensor([cat_values], dtype=torch.long)
    cont_tensor = torch.tensor(cont_values, dtype=torch.float32)

    with torch.no_grad():
        output = model(cat_tensor, cont_tensor)
        predicted_class = output.argmax(dim=1).item()

    return ">50K" if predicted_class == 1 else "<=50K"


In [ ]:
# Example prediction
result = predict_income(
    age=45,
    sex="Male",
    education="Bachelors",
    education_num=13,
    marital_status="Married",
    workclass="Private",
    occupation="Exec-managerial",
    hours_per_week=45
)

print("Predicted income:", result)


## 13. Conclusion

The PyTorch neural network successfully performed binary classification on the Census Income Dataset.

### Final Result
- **Training samples:** 25,000
- **Testing samples:** 5,000
- **Epochs:** 300
- **Hidden neurons:** 50
- **Dropout:** 0.4
- **Optimizer:** Adam
- **Learning rate:** 0.001
- **Test Loss:** approximately **0.2670**
- **Test Accuracy:** approximately **88.02%**

The model combines categorical embeddings with batch-normalized continuous features to predict whether income is `<=50K` or `>50K`.